In [ ]:
import sys
sys.path.append('../../../code/libs/')

%load_ext autoreload
%autoreload 2
import utils
import viz
import ios
import constants
import text as txtlib

In [2]:
# import os
import pandas as pd
import numpy as np

In [3]:
ROOT = 'population_2015_2022/'
metric = 'population'
metricy = 'years'
COLS = {'NAME':'state_name',
        'S0101_C01_001E': metric,
        'S0101_C03_001E': f'{metric}_males',
        'S0101_C05_001E': f'{metric}_females',
        'S0101_C01_002E': f'{metricy}_under_5', 
        'S0101_C01_003E': f'{metricy}_5_to_9', 
        'S0101_C01_004E': f'{metricy}_10_to_14',
        'S0101_C01_005E': f'{metricy}_15_to_19',
        'S0101_C01_006E': f'{metricy}_20_to_24',
        'S0101_C01_007E': f'{metricy}_25_to_29',
        'S0101_C01_008E': f'{metricy}_30_to_34',
        'S0101_C01_009E': f'{metricy}_35_to_39',
        'S0101_C01_010E': f'{metricy}_40_to_44',
        'S0101_C01_011E': f'{metricy}_45_to_49',
        'S0101_C01_012E': f'{metricy}_50_to_54',
        'S0101_C01_013E': f'{metricy}_55_to_59',
        'S0101_C01_014E': f'{metricy}_60_to_64',
        'S0101_C01_015E': f'{metricy}_65_to_69',
        'S0101_C01_016E': f'{metricy}_70_to_74',
        'S0101_C01_017E': f'{metricy}_75_to_79',
        'S0101_C01_018E': f'{metricy}_80_to_84',
        'S0101_C01_019E': f'{metricy}_85_and_above'}

In [4]:
# state_name	po_total	po_male	
ios.get_files_from_pattern(ios.path_join(ROOT,"*5Y*-Data.csv"))

['population_2015_2022/ACSST5Y2015.S0101-Data.csv',
 'population_2015_2022/ACSST5Y2016.S0101-Data.csv',
 'population_2015_2022/ACSST5Y2017.S0101-Data.csv',
 'population_2015_2022/ACSST5Y2018.S0101-Data.csv',
 'population_2015_2022/ACSST5Y2019.S0101-Data.csv',
 'population_2015_2022/ACSST5Y2020.S0101-Data.csv',
 'population_2015_2022/ACSST5Y2021.S0101-Data.csv',
 'population_2015_2022/ACSST5Y2022.S0101-Data.csv']

In [5]:
files = ios.get_files_from_pattern(ios.path_join(ROOT,"*5Y*-Data.csv"))
data = pd.DataFrame()

for fn in files:
    year = int(fn.split('.')[0].split('Y')[-1])
    tmp = ios.read_csv(fn)
    
    cols = COLS.copy()
    if year in [2015,2016]:
        cols['S0101_C02_001E'] = cols['S0101_C03_001E']  # females
        cols['S0101_C03_001E'] = cols['S0101_C05_001E']  # males
        del(cols['S0101_C05_001E'])
        C = 3
    else:
        C = 5

    
    print(year)
    
    # population feamle reproductive age (15-49)
    cols_ra = [f'S0101_C0{C}_0{str(i).zfill(2)}E' for i in range(5,12)]
    
    all_cols = sorted(cols.keys()) + cols_ra 
    tmp = tmp[all_cols]
    tmp.rename(columns=cols, inplace=True)
    tmp.drop('Geography', inplace=True)
    tmp.loc[:,'year'] = year
    tmp.set_index(['state_name','year'], inplace=True)
    tmp = tmp.astype(float)
    tmp_ra = tmp.copy()
    tmp.drop(columns=cols_ra, inplace=True)
    tmp_ra.drop(columns=[c for c in COLS.values() if c not in ['state_name']], inplace=True)
    
    # add reproductive age (15-49 females)
    tmp_ra = pd.DataFrame(tmp_ra.sum(axis=1), columns=['pop_females_reproductive'])
    tmp = tmp.join(tmp_ra)
    
    data = pd.concat([data, tmp], ignore_index=False)

2015
2016
2017
2018
2019
2020
2021
2022


In [6]:
data = data[['population','population_females','population_males'] + [c for c in data.columns if c.startswith('years_')] + ['pop_females_reproductive']]
data

,,population,population_females,population_males,years_under_5,years_5_to_9,years_10_to_14,years_15_to_19,years_20_to_24,years_25_to_29,years_30_to_34,...,years_45_to_49,years_50_to_54,years_55_to_59,years_60_to_64,years_65_to_69,years_70_to_74,years_75_to_79,years_80_to_84,years_85_and_above,pop_females_reproductive
state_name,year,,,,,,,,,,,,,,,,,,,,,
Alabama,2015,4830620.0,2489527.0,2341093.0,6.1,6.3,6.6,6.7,7.2,6.5,6.4,...,6.6,7.1,6.8,6.2,5.0,3.6,2.7,1.9,1.7,45.2
Alaska,2015,733375.0,349215.0,384160.0,7.5,7.0,7.0,6.9,8.3,8.3,7.3,...,6.5,7.2,7.0,5.6,3.6,2.3,1.3,0.9,0.7,48.6
Arizona,2015,6641928.0,3342840.0,3299088.0,6.5,6.9,6.9,6.9,7.2,6.7,6.6,...,6.2,6.5,6.1,5.7,5.0,3.9,2.8,1.9,1.8,45.0
Arkansas,2015,2958208.0,1506295.0,1451913.0,6.5,6.8,6.6,6.7,7.0,6.5,6.5,...,6.4,6.8,6.5,5.9,5.0,3.8,2.7,2.0,1.8,44.4
California,2015,38421464.0,19334329.0,19087135.0,6.5,6.6,6.6,6.9,7.6,7.5,7.2,...,6.8,6.9,6.2,5.3,4.1,2.9,2.1,1.6,1.7,48.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Virginia,2022,8624511.0,4355736.0,4268775.0,494148.0,511965.0,545595.0,573642.0,580019.0,579897.0,590216.0,...,541770.0,561174.0,576469.0,543459.0,453677.0,365967.0,251265.0,158796.0,151301.0,1978555.0
Washington,2022,7688549.0,3810631.0,3877918.0,440172.0,465882.0,477398.0,464142.0,493641.0,571263.0,592493.0,...,467334.0,466904.0,471024.0,484289.0,414287.0,336203.0,210852.0,130016.0,137582.0,1762040.0
West Virginia,2022,1792967.0,898195.0,894772.0,90380.0,98563.0,106128.0,112676.0,113691.0,107037.0,103882.0,...,111972.0,116126.0,120980.0,129971.0,121314.0,98411.0,65569.0,41634.0,39516.0,370780.0


In [7]:
ios.save_csv(data, 'population_2015_2022.csv')

In [8]:
for year, tmp in data.reset_index().groupby("year"):
    print(year, tmp.population.sum())

2015 316515021.0
2016 318558162.0
2017 321004407.0
2018 322903030.0
2019 324697795.0
2020 326569308.0
2021 329725481.0
2022 331097593.0


In [9]:
for year, tmp in data.reset_index().groupby("year"):
    print(year, tmp[[c for c in data.columns if c.startswith('years_')]].sum().sum())

2015 5099.3
2016 5099.2
2017 321004407.0
2018 322903030.0
2019 324697795.0
2020 326569308.0
2021 329725481.0
2022 331097593.0


In [10]:
data.query("year == 2022").population.sum()

331097593.0